In [ ]:
import cv2
import numpy as np
from scipy.spatial import distance
import mediapipe as mp
import time
from collections import deque

try:
    from IPython.display import display, Image, clear_output
    JUPYTER_MODE = True
except ImportError:
    JUPYTER_MODE = False

print("✓ All libraries imported successfully!")


In [ ]:
EAR_THRESHOLD = 0.25  # Eye Aspect Ratio threshold for closed eyes
MAR_THRESHOLD = 0.6   # Mouth Aspect Ratio threshold for yawning
EAR_CONSEC_FRAMES = 20  # Consecutive frames for drowsiness alert
MAR_CONSEC_FRAMES = 15  # Consecutive frames for yawn detection

# MediaPipe Face Mesh landmark indices
LEFT_EYE_INDICES = [33, 160, 158, 133, 153, 144]  # Left eye landmarks
RIGHT_EYE_INDICES = [362, 385, 387, 263, 373, 380]  # Right eye landmarks
MOUTH_INDICES = [61, 291, 0, 17, 84, 181, 78, 82, 13, 312, 311, 308, 324, 318]  # Mouth landmarks

print("✓ Constants defined:")
print(f"  - EAR Threshold: {EAR_THRESHOLD}")
print(f"  - MAR Threshold: {MAR_THRESHOLD}")
print(f"  - Alert after {EAR_CONSEC_FRAMES} consecutive frames")

In [ ]:
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Create Face Mesh instance
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print("✓ MediaPipe Face Mesh initialized")
print("  Using 478-point facial landmark model")

In [ ]:
def eye_aspect_ratio(eye_landmarks):
    """
    Calculate Eye Aspect Ratio (EAR)
    Formula: EAR = (||p2-p6|| + ||p3-p5||) / (2 * ||p1-p4||)
    """
    # Vertical eye distances
    A = distance.euclidean(eye_landmarks[1], eye_landmarks[5])
    B = distance.euclidean(eye_landmarks[2], eye_landmarks[4])
    
    # Horizontal eye distance
    C = distance.euclidean(eye_landmarks[0], eye_landmarks[3])
    
    # EAR formula
    ear = (A + B) / (2.0 * C)
    return ear


def mouth_aspect_ratio(mouth_landmarks):
    """
    Calculate Mouth Aspect Ratio (MAR)
    Higher MAR indicates open mouth (yawning)
    """
    # Calculate vertical distances
    vertical_dists = []
    for i in range(1, len(mouth_landmarks) // 2):
        dist = distance.euclidean(mouth_landmarks[i], 
                                  mouth_landmarks[len(mouth_landmarks) - i])
        vertical_dists.append(dist)
    
    # Calculate horizontal distance
    horizontal_dist = distance.euclidean(mouth_landmarks[0], 
                                        mouth_landmarks[len(mouth_landmarks) // 2])
    
    # MAR formula
    mar = np.mean(vertical_dists) / horizontal_dist if horizontal_dist > 0 else 0
    return mar

def get_landmarks(face_landmarks, indices, frame_width, frame_height):
    """Extract specific landmarks and convert to pixel coordinates"""
    landmarks = []
    for idx in indices:
        landmark = face_landmarks.landmark[idx]
        x = int(landmark.x * frame_width)
        y = int(landmark.y * frame_height)
        landmarks.append([x, y])
    return np.array(landmarks)

print("✓ Aspect ratio functions defined")

In [ ]:
def draw_landmarks(frame, landmarks, color, thickness=2):
    """Draw landmarks on the frame"""
    if len(landmarks) > 0:
        points = landmarks.reshape((-1, 1, 2))
        cv2.polylines(frame, [points], True, color, thickness)


def draw_text_with_background(frame, text, position, font_scale=0.6, 
                              thickness=2, text_color=(255, 255, 255), 
                              bg_color=(0, 0, 0)):
    """Draw text with background for better visibility"""
    font = cv2.FONT_HERSHEY_SIMPLEX
    (text_width, text_height), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    
    x, y = position
    # Draw background rectangle
    cv2.rectangle(frame, (x, y - text_height - 5), 
                 (x + text_width + 5, y + 5), bg_color, -1)
    # Draw text
    cv2.putText(frame, text, (x, y), font, font_scale, text_color, thickness)

print("✓ Drawing functions defined")

In [ ]:
class DrowsinessDetector:
    def __init__(self):
        """Initialize the drowsiness detector"""
        # Counters and history
        self.eye_closed_frames = 0
        self.yawn_frames = 0
        self.total_blinks = 0
        self.total_yawns = 0
        self.drowsy_count = 0
        
        # History buffers
        self.ear_history = deque(maxlen=30)
        self.mar_history = deque(maxlen=30)
        
        # Timestamps
        self.last_blink_time = time.time()
        self.alert_start_time = None
        
        print("✓ DrowsinessDetector initialized successfully")
    
    def process_frame(self, frame):
        """Process a single frame and detect drowsiness"""
        frame_height, frame_width = frame.shape[:2]
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Process frame with MediaPipe
        results = face_mesh.process(rgb_frame)
        
        if not results.multi_face_landmarks:
            cv2.putText(frame, "No Face Detected", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            return frame, False, {}
        
        face_landmarks = results.multi_face_landmarks[0]
        
        # Extract eye and mouth landmarks
        left_eye = get_landmarks(face_landmarks, LEFT_EYE_INDICES, frame_width, frame_height)
        right_eye = get_landmarks(face_landmarks, RIGHT_EYE_INDICES, frame_width, frame_height)
        mouth = get_landmarks(face_landmarks, MOUTH_INDICES, frame_width, frame_height)
        
        # Calculate EAR for both eyes
        left_ear = eye_aspect_ratio(left_eye)
        right_ear = eye_aspect_ratio(right_eye)
        ear = (left_ear + right_ear) / 2.0
        
        # Calculate MAR
        mar = mouth_aspect_ratio(mouth)
        
        # Store history
        self.ear_history.append(ear)
        self.mar_history.append(mar)
        
        # Drowsiness detection
        drowsy = False
        
        # Eye closure detection
        if ear < EAR_THRESHOLD:
            self.eye_closed_frames += 1
            
            if self.eye_closed_frames >= EAR_CONSEC_FRAMES:
                drowsy = True
                self.drowsy_count += 1
                
                # Draw alert
                cv2.putText(frame, "DROWSINESS ALERT!", (10, 50),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
                
                if self.alert_start_time is None:
                    self.alert_start_time = time.time()
        else:
            # Blink detection
            if self.eye_closed_frames > 3 and self.eye_closed_frames < EAR_CONSEC_FRAMES:
                current_time = time.time()
                if current_time - self.last_blink_time > 0.3:
                    self.total_blinks += 1
                    self.last_blink_time = current_time
            
            self.eye_closed_frames = 0
            self.alert_start_time = None
        
        # Yawn detection
        if mar > MAR_THRESHOLD:
            self.yawn_frames += 1
            
            if self.yawn_frames >= MAR_CONSEC_FRAMES:
                cv2.putText(frame, "YAWN DETECTED!", (10, 90),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 165, 255), 3)
        else:
            if self.yawn_frames >= MAR_CONSEC_FRAMES:
                self.total_yawns += 1
            self.yawn_frames = 0
        
        # Draw facial features
        eye_color = (0, 0, 255) if ear < EAR_THRESHOLD else (0, 255, 0)
        mouth_color = (0, 165, 255) if mar > MAR_THRESHOLD else (255, 0, 0)
        
        draw_landmarks(frame, left_eye, eye_color, 2)
        draw_landmarks(frame, right_eye, eye_color, 2)
        draw_landmarks(frame, mouth, mouth_color, 2)
        
        # Calculate metrics
        avg_ear = np.mean(self.ear_history) if self.ear_history else 0
        avg_mar = np.mean(self.mar_history) if self.mar_history else 0
        
        # Display metrics panel
        panel_height = 140
        y_offset = frame_height - panel_height - 10
        cv2.rectangle(frame, (10, y_offset), (320, frame_height - 10), (0, 0, 0), -1)
        cv2.rectangle(frame, (10, y_offset), (320, frame_height - 10), (255, 255, 255), 2)
        
        metrics_text = [
            f"EAR: {ear:.3f} (Avg: {avg_ear:.3f})",
            f"MAR: {mar:.3f} (Avg: {avg_mar:.3f})",
            f"Blinks: {self.total_blinks}",
            f"Yawns: {self.total_yawns}"
        ]
        
        for i, text in enumerate(metrics_text):
            y_pos = y_offset + 30 + (i * 28)
            cv2.putText(frame, text, (20, y_pos),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Drowsiness indicator bar
        drowsiness_level = min(100, (self.drowsy_count * 5))
        bar_width = int(300 * drowsiness_level / 100)
        
        # Color based on level
        if drowsiness_level < 30:
            bar_color = (0, 255, 0)  # Green
            status = "ALERT"
        elif drowsiness_level < 60:
            bar_color = (0, 165, 255)  # Orange
            status = "CAUTION"
        else:
            bar_color = (0, 0, 255)  # Red
            status = "DROWSY"
        
        # Draw bar background
        bar_x = frame_width - 320
        cv2.rectangle(frame, (bar_x, 20), (frame_width - 20, 60), (50, 50, 50), -1)
        cv2.rectangle(frame, (bar_x, 20), (frame_width - 20, 60), (255, 255, 255), 2)
        
        # Draw bar fill
        if bar_width > 0:
            cv2.rectangle(frame, (bar_x, 20), (bar_x + bar_width, 60), bar_color, -1)
        
        # Draw text
        cv2.putText(frame, f"{status}: {drowsiness_level}%", (bar_x + 10, 45),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # Create stats dictionary
        stats = {
            'ear': ear,
            'mar': mar,
            'avg_ear': avg_ear,
            'avg_mar': avg_mar,
            'blinks': self.total_blinks,
            'yawns': self.total_yawns,
            'drowsiness': drowsiness_level,
            'drowsy': drowsy,
            'status': status
        }
        
        return frame, drowsy, stats
    
    def reset(self):
        """Reset all counters"""
        self.eye_closed_frames = 0
        self.yawn_frames = 0
        self.total_blinks = 0
        self.total_yawns = 0
        self.drowsy_count = 0
        self.ear_history.clear()
        self.mar_history.clear()
        print("✓ Counters reset!")

print("✓ DrowsinessDetector class defined")

In [ ]:
detector = DrowsinessDetector()

print("\n" + "="*60)
print("✓ Detector ready! Proceed to the next cell to start detection.")
print("="*60)

In [ ]:
"""
Run the drowsiness detector with OpenCV window
Press 'q' to quit, 'r' to reset counters, 's' to save screenshot
"""

def run_detector_opencv():
    """Run detector using OpenCV window (best for local Jupyter)"""
    # Initialize webcam
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    if not cap.isOpened():
        print("❌ Error: Could not open webcam")
        return
    
    print("✓ Camera initialized!")
    print("Controls:")
    print("  - Press 'q' to quit")
    print("  - Press 'r' to reset counters")
    print("  - Press 's' to save screenshot")
    print("\nDetecting drowsiness...\n")
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("❌ Failed to capture frame")
                break
            
            # Process frame
            frame, drowsy, stats = detector.process_frame(frame)
            
            # Display frame
            cv2.imshow('Micro-Drowsiness Detector - Press Q to Quit', frame)
            
            # Handle keyboard input
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                print("\nQuitting...")
                break
            elif key == ord('r'):
                detector.reset()
            elif key == ord('s'):
                timestamp = time.strftime("%Y%m%d-%H%M%S")
                filename = f"drowsiness_capture_{timestamp}.jpg"
                cv2.imwrite(filename, frame)
                print(f"📸 Screenshot saved: {filename}")
    
    except KeyboardInterrupt:
        print("\nInterrupted by user")
    
    finally:
        # Cleanup
        cap.release()
        cv2.destroyAllWindows()
        
        print("\n" + "="*60)
        print("📊 SESSION SUMMARY")
        print("="*60)
        print(f"  Total Blinks: {detector.total_blinks}")
        print(f"  Total Yawns: {detector.total_yawns}")
        print(f"  Drowsiness Alerts: {detector.drowsy_count}")
        print("="*60)

In [ ]:
"""
Run the drowsiness detector with inline display in Jupyter
Better for Colab or online Jupyter environments
"""

def run_detector_jupyter(duration=60, display_interval=5):
    """
    Run detector with inline display in Jupyter
    
    Args:
        duration: How many seconds to run (default: 60)
        display_interval: Update display every N frames (default: 5)
    """
    # Initialize webcam
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    if not cap.isOpened():
        print("❌ Error: Could not open webcam")
        return
    
    print("✓ Camera initialized!")
    print(f"🎥 Running detection for {duration} seconds...")
    print("📊 Processing frames...\n")
    
    start_time = time.time()
    frame_count = 0
    
    try:
        while (time.time() - start_time) < duration:
            ret, frame = cap.read()
            if not ret:
                print("❌ Failed to capture frame")
                break
            
            # Process frame
            frame, drowsy, stats = detector.process_frame(frame)
            frame_count += 1
            
            # Display at intervals
            if frame_count % display_interval == 0:
                # Convert BGR to RGB for display
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                _, buffer = cv2.imencode('.jpg', frame_rgb, [cv2.IMWRITE_JPEG_QUALITY, 85])
                
                if JUPYTER_MODE:
                    clear_output(wait=True)
                    display(Image(data=buffer.tobytes()))
                
                # Print stats
                elapsed = int(time.time() - start_time)
                print(f"⏱️  Time: {elapsed}s / {duration}s | Frame: {frame_count}")
                print(f"👁️  EAR: {stats['ear']:.3f} (Avg: {stats['avg_ear']:.3f})")
                print(f"👄 MAR: {stats['mar']:.3f} (Avg: {stats['avg_mar']:.3f})")
                print(f"😑 Blinks: {stats['blinks']} | 🥱 Yawns: {stats['yawns']}")
                print(f"💤 Status: {stats['status']} ({stats['drowsiness']}%)")
                
                if stats['drowsy']:
                    print("⚠️  🚨 DROWSINESS ALERT! 🚨")
                print("-" * 60)
    
    except KeyboardInterrupt:
        print("\n⚠️  Interrupted by user")
    
    finally:
        # Cleanup
        cap.release()
        
        elapsed_time = int(time.time() - start_time)
        print("\n" + "="*60)
        print("📊 SESSION SUMMARY")
        print("="*60)
        print(f"  Duration: {elapsed_time}s")
        print(f"  Frames Processed: {frame_count}")
        print(f"  Total Blinks: {detector.total_blinks}")
        print(f"  Total Yawns: {detector.total_yawns}")
        print(f"  Drowsiness Alerts: {detector.drowsy_count}")
        print("="*60)

In [ ]:
"""
Real-time video capture with live updates in Jupyter
This is the BEST option for Jupyter notebooks!
"""

def run_realtime_detection(stop_button_text="Stop Detection"):
    """
    Real-time drowsiness detection with interactive controls
    Shows live video feed in Jupyter with automatic updates
    """
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    from threading import Thread
    
    # Create output widget for video
    output = widgets.Output()
    
    # Create control buttons
    stop_button = widgets.Button(
        description=stop_button_text,
        button_style='danger',
        icon='stop'
    )
    
    reset_button = widgets.Button(
        description='Reset Counters',
        button_style='warning',
        icon='refresh'
    )
    
    # Status display
    status_label = widgets.HTML(value="<h3>Initializing camera...</h3>")
    
    # Metrics display
    metrics_html = widgets.HTML(value="")
    
    # Layout
    button_box = widgets.HBox([stop_button, reset_button])
    display(status_label)
    display(button_box)
    display(metrics_html)
    display(output)
    
    # Control flag
    running = {'value': True}
    
    def stop_detection(b):
        running['value'] = False
        stop_button.description = "Stopping..."
        stop_button.disabled = True
    
    def reset_counters(b):
        detector.reset()
        status_label.value = "<h3 style='color: green;'>✓ Counters Reset!</h3>"
    
    stop_button.on_click(stop_detection)
    reset_button.on_click(reset_counters)
    
    # Video capture thread
    def capture_loop():
        cap = cv2.VideoCapture(0)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        cap.set(cv2.CAP_PROP_FPS, 30)
        
        if not cap.isOpened():
            status_label.value = "<h3 style='color: red;'>❌ Could not open camera</h3>"
            return
        
        status_label.value = "<h3 style='color: green;'>🎥 Camera Active - Detecting Drowsiness</h3>"
        
        frame_count = 0
        start_time = time.time()
        
        try:
            while running['value']:
                ret, frame = cap.read()
                if not ret:
                    break
                
                # Process frame
                frame, drowsy, stats = detector.process_frame(frame)
                frame_count += 1
                
                # Update display every 2 frames for smooth performance
                if frame_count % 2 == 0:
                    # Convert to RGB and encode
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    _, buffer = cv2.imencode('.jpg', frame_rgb, [cv2.IMWRITE_JPEG_QUALITY, 80])
                    
                    # Update video output
                    with output:
                        clear_output(wait=True)
                        display(Image(data=buffer.tobytes()))
                    
                    # Calculate FPS
                    elapsed = time.time() - start_time
                    fps = frame_count / elapsed if elapsed > 0 else 0
                    
                    # Update metrics
                    status_color = "red" if stats['drowsy'] else "green"
                    alert_text = "🚨 DROWSINESS ALERT! 🚨" if stats['drowsy'] else ""
                    
                    metrics_html.value = f"""
                    <div style='background-color: #f0f0f0; padding: 15px; border-radius: 10px; font-family: monospace;'>
                        <h3 style='margin-top: 0; color: {status_color};'>
                            Status: {stats['status']} ({stats['drowsiness']}%) {alert_text}
                        </h3>
                        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 10px;'>
                            <div>
                                <b>👁️ Eye Aspect Ratio:</b> {stats['ear']:.3f}<br>
                                <b>📊 Average EAR:</b> {stats['avg_ear']:.3f}<br>
                                <b>😑 Total Blinks:</b> {stats['blinks']}
                            </div>
                            <div>
                                <b>👄 Mouth Aspect Ratio:</b> {stats['mar']:.3f}<br>
                                <b>📊 Average MAR:</b> {stats['avg_mar']:.3f}<br>
                                <b>🥱 Total Yawns:</b> {stats['yawns']}
                            </div>
                        </div>
                        <div style='margin-top: 10px;'>
                            <b>⚡ FPS:</b> {fps:.1f} | <b>📹 Frames:</b> {frame_count} | <b>⏱️ Time:</b> {int(elapsed)}s
                        </div>
                    </div>
                    """
        
        except Exception as e:
            status_label.value = f"<h3 style='color: red;'>❌ Error: {str(e)}</h3>"
        
        finally:
            cap.release()
            elapsed_time = time.time() - start_time
            
            status_label.value = f"""
            <h3 style='color: blue;'>✓ Detection Stopped</h3>
            <p><b>Session Duration:</b> {int(elapsed_time)}s | <b>Total Frames:</b> {frame_count}</p>
            """
            
            stop_button.description = "Stopped"
            stop_button.button_style = 'success'
    
    # Start capture in thread
    thread = Thread(target=capture_loop)
    thread.daemon = True
    thread.start()

print("✓ Real-time detection function ready!")
print("\nTo start real-time detection, run:")
print(">>> run_realtime_detection()")